# mine-tuning-model (RTX 5060 Ti 최적화 버전)

## 사전 준비 (터미널에서 먼저 실행)

```bash
# 1. .env 파일에 HF 토큰 설정
cp docker/.env.example docker/.env
# docker/.env 열고 HF_TOKEN 입력

# 2. SGLang 서버 실행
cd docker
docker compose up -d

# 3. 서버 로그 확인 (선택)
docker logs -f sglang-server
```

> 💡 처음 실행 시 Qwen3-8B 모델 다운로드로 약 16GB 다운로드가 필요해요!

## 0. GPU 확인 (제일 먼저 실행!)

In [23]:
import subprocess
result = subprocess.run(
    ['docker', 'compose', 'up', '-d'],
    capture_output=True,
    text=True,
    cwd=r'C:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\docker'  # docker 폴더 절대경로
)
print(result.stdout)
print(result.stderr)


 Container sglang-server Creating 
 Container sglang-server Created 
 Container sglang-server Starting 
 Container sglang-server Started 



In [24]:
import requests
import time

print("⏳ SGLang 서버 연결 확인 중...")

try:
    r = requests.get("http://localhost:30000/health")
    if r.status_code == 200:
        print("SGLang 서버 정상 동작 중!")
except:
    print('연결 실패')

⏳ SGLang 서버 연결 확인 중...
SGLang 서버 정상 동작 중!


## 1. 필수 라이브러리 설치

> 💡 처음 한 번만 실행하면 돼요!

In [25]:
import subprocess
subprocess.run(['pip', 'install', 'requests', '-q'], check=True)
print("완료")

완료


In [26]:
import wandb

wandb.login()

print('✅ wandb 로그인 완료')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\SSAFY\_netrc.
wandb: Currently logged in as: ddorin789 (ddorin789-nothing) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ wandb 로그인 완료


## 2. 데이터 로드 및 Instruction 형식 변환

In [27]:
from datasets import load_dataset

# 데이터 로드
ds = load_dataset('ddorin/minecraft-question-answer-260k')
print(ds)

# Instruction 형식으로 변환
def format_instruction(example):
    return {
        'text': f"""<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
{example['question']}
<|im_end|>
<|im_start|>assistant
{example['answer']}
<|im_end|>"""
    }

train_data = ds['train'].map(format_instruction, remove_columns=['question', 'answer', 'source'])
print(f'✅ 데이터 변환 완료: {len(train_data)}개')
print('\n샘플 확인:')
print(train_data[0]['text'])

c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'source'],
        num_rows: 268239
    })
})
✅ 데이터 변환 완료: 268239개

샘플 확인:
<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
What types of leaves can be found naturally in jungle bushes in Minecraft?
<|im_end|>
<|im_start|>assistant
In Minecraft, jungle bushes can generate oak leaves in the Java Edition and jungle leaves in the Bedrock Edition.
<|im_end|>


## 3.추론 함수

In [ ]:
import requests
import json

SGLANG_URL = "http://localhost:30000/v1/chat/completions"

def chat(prompt, thinking=False):
    payload = {
        "model": "Qwen/Qwen3-8B",
        "messages": [
            {"role": "system", "content": "You are a helpful Minecraft expert assistant."},
            {"role": "user",   "content": prompt}
        ],
        "temperature": 0.6,
        "top_p": 0.95,
        "max_tokens": 512,
        "extra_body": {"separate_reasoning": thinking}
    }
    resp = requests.post(SGLANG_URL, json=payload)
    data = resp.json()

    if thinking:
        reasoning = data["choices"][0]["message"].get("reasoning_content", "")
        content   = data["choices"][0]["message"]["content"]
        print("🧠 [Thinking]\n", reasoning)
        print("\n💬 [Answer]\n", content)
    else:
        print(data["choices"][0]["message"]["content"])

print("✅ 추론 함수 준비 완료")

## 기본 추론 테스트

In [29]:
import requests
import json

SGLANG_URL = "http://localhost:30000/v1/chat/completions"

# 추론 테스트 함수 작성
def chat(prompt, thinking=False):
    payload = {
        "model": "Qwen/Qwen3-8B",
        "messages": [
            {"role": "system", "content": "You are a helpful Minecraft expert assistant."},
            {"role": "user",   "content": prompt}
        ],
        "temperature": 0.6,
        "top_p": 0.95,
        "max_tokens": 512,
        "extra_body": {"separate_reasoning": thinking}
    }
    resp = requests.post(SGLANG_URL, json=payload)
    data = resp.json()

    if thinking:
        reasoning = data["choices"][0]["message"].get("reasoning_content", "")
        content   = data["choices"][0]["message"]["content"]
        print("🧠 [Thinking]\n", reasoning)
        print("\n💬 [Answer]\n", content)
    else:
        print(data["choices"][0]["message"]["content"])

In [ ]:
# 기본 추론 테스트
chat("다이아몬드는 어디서 가장 잘 나와?")

## 5. 학습 실행 (SFTTrainer)

### ⚙️ 매 세션마다 아래 변수만 수정하세요

In [ ]:
from trl import SFTTrainer, SFTConfig
import os

# ============================================================
# 전체 데이터 학습 설정
# ============================================================
OUTPUT_DIR = r"C:\mine-tuning-output\full_train"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📦 전체 학습 데이터: {len(train_data):,}건")

# wandb 실험 초기화
wandb.init(
    project="minecraft-ft-qwen2.5-7b",  # wandb 프로젝트 이름
    name="full_train_3epoch",            # 실험 이름
    config={
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "dataset": "ddorin/minecraft-question-answer-260k",
        "epochs": 3,
        "batch_size": 4,
        "lora_r": 16,
        "lora_alpha": 32,
        "optim": "paged_adamw_8bit",
    }
)

# 학습 설정
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                 # 전체 데이터 3회 학습
    per_device_train_batch_size=4,      # RTX 5060 Ti 16GB 기준
    gradient_accumulation_steps=4,      # 실질 배치 크기 = 16
    learning_rate=2e-4,
    bf16=True,                          # RTX 5060 Ti 권장
    fp16=False,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="wandb",                  # ✅ wandb 로깅
    dataloader_num_workers=0,           # Windows 필수
    optim="paged_adamw_8bit",
)

# Trainer 생성
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,           # ✅ 전체 데이터 사용
    args=training_args,
    processing_class=tokenizer,
)

print("🚀 전체 데이터 학습 시작!")
trainer.train()

# 최종 저장
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ 전체 학습 완료! → {OUTPUT_DIR}")

# wandb 실험 종료
wandb.finish()
print("✅ wandb 로깅 완료")

## 6. 버전 확인

In [ ]:
import transformers, trl, torch

print(f'transformers: {transformers.__version__}')
print(f'trl:          {trl.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CUDA:         {torch.version.cuda}')

---

## 📋 RTX 5060 Ti 실행 시 주의사항

| 항목 | 내용 |
|------|------|
| **PyTorch** | CUDA 12.8 버전으로 설치 (`--index-url .../cu128`) |
| **bf16** | RTX 5060 Ti (Blackwell)는 `bf16=True` 사용 가능 ✅ |
| **fp16** | bf16 사용 시 반드시 `fp16=False` |
| **bitsandbytes** | `pip install bitsandbytes` (최신 버전 Windows 공식 지원) |
| **dataloader_num_workers** | 반드시 `0`으로 설정 (Windows 멀티프로세싱 이슈) |
| **경로** | `OUTPUT_DIR`에 한글/공백이 없는 경로 사용 권장 |
| **VRAM 부족 시** | `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` |

### 🐛 자주 나오는 에러

```
CUDA error: no kernel image is available for execution on the device
```
→ PyTorch가 5060 Ti를 지원하지 않는 버전이에요. cu128 버전으로 재설치하세요.

```
bitsandbytes: CUDA Setup failed
```
→ `pip install bitsandbytes --upgrade` 로 최신 버전으로 업그레이드하세요.